# Decision-aligned quality-preserving LLM routing

This notebook is the executable v4 companion to
`llm_router_project_scope_4.md`. It does **not** regenerate candidate
responses. Instead, it reuses the complete, fingerprinted v3 evidence and
asks a cleaner decision question:

> When can a faster candidate replace the strongest model without reducing
> observed answer quality?

The walkthrough improves five parts of v3:

1. safety labels are relative to the deployed fallback;
2. the loss penalizes missed safe speedups and unsafe switches differently;
3. Platt calibration learns both a slope and an intercept per candidate;
4. each candidate receives its own validation-selected safety threshold; and
5. every checkpoint is judged with calibrated probabilities and measured
   batch-one router overhead—the same rule used for final deployment.

```mermaid
flowchart LR
    A["Reused v3 evidence"] --> B["Fallback-relative labels"]
    B --> C["ModernBERT + rank-4 LoRA"]
    C --> D["Safety / latency / token heads"]
    D --> E["Out-of-fold Platt calibration"]
    E --> F["Per-candidate thresholds"]
    F --> G["Overhead-inclusive validation guard"]
    G --> H["Sealed test or fallback-only deployment"]
```

## Run instructions

1. Complete notebook 03 first so its v3 Drive files exist.
2. Select the same GPU type recorded by the v3 measurement manifest.
3. Run the install cell, restart the runtime, and run top to bottom.

A disabled router remains a valid result: it means no validation-safe speedup
was demonstrated.


In [1]:
# Install the router-training runtime. Candidate-generation dependencies are
# not needed because this notebook consumes v3 parquet measurements.
%pip install -q -U "transformers==4.53.1" \
    "huggingface-hub==0.36.2" accelerate pyarrow scikit-learn \
    "peft==0.17.1" "ipywidgets>=8"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 46.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-h

## 1. Synchronized v4 contract

The contract below is intentionally explicit. Any setting that can change
training, calibration, selection, or online overhead belongs in the v4
router fingerprint.

Candidate evidence keeps its original v3 generation fingerprint. Router
experiments receive a separate v4 fingerprint, so they can reuse expensive
generations without overwriting v3 artifacts.

**Expected output:** the reused evidence tag, new router tag, current GPU,
and output directories.


In [2]:
from __future__ import annotations

import copy, gc, hashlib, html as html_lib, itertools, json, math, os
import random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import peft
import sklearn
import torch
import torch.nn.functional as F
import transformers
from IPython.display import HTML, clear_output, display
from sklearn.metrics import brier_score_loss, mean_absolute_error, r2_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader

# --- Synchronized identity and evidence requirements ---
V4_SCOPE_SCHEMA_VERSION = 4
V4_SCOPE_FILE_NAME = "llm_router_project_scope_4.md"
REQUIRED_V3_SCHEMA_VERSION = 3
REQUIRED_V3_PROMPT_TEMPLATE = "v3-json-only-2026-08-07"
SEED = 42
TASKS = ("gsm8k", "mmlu", "arc_challenge")
N_PER_TASK = 300
MODEL_NAMES = (
    "qwen2.5-1.5b-ar",
    "fast-dllm-v2-1.5b",
    "qwen2.5-7b-4bit",
)
EXPECTED_MODEL_REPOS = {
    "qwen2.5-1.5b-ar": "Qwen/Qwen2.5-1.5B-Instruct",
    "fast-dllm-v2-1.5b": "Efficient-Large-Model/Fast_dLLM_v2_1.5B",
    "qwen2.5-7b-4bit": "Qwen/Qwen2.5-7B-Instruct",
}

# --- Quality and deployment constraints ---
MIN_QUALITY_RETENTION = 0.98
QUALITY_SAFETY_EPSILON = 0.0
MIN_PREDICTED_SPEEDUP = 0.02
SAFETY_DEFINITION = "candidate_quality >= fallback_quality - epsilon"

# --- Smaller, parameter-efficient router ---
ROUTER_ENCODER_REPO = "nomic-ai/modernbert-embed-base"
ROUTER_ENCODER_REVISION = "d556a88e332558790b210f7bdbe87da2fa94a8d8"
ROUTER_MAX_INPUT_TOKENS = 512
ROUTER_LORA_R = 4
ROUTER_LORA_ALPHA = 8
ROUTER_LORA_DROPOUT = 0.05
ROUTER_LORA_TARGET_MODULES = "all-linear"
ROUTER_BATCH_SIZE = 8
ROUTER_GRADIENT_ACCUMULATION = 2
ROUTER_MAX_EPOCHS = 15
ROUTER_MIN_EPOCHS = 6
ROUTER_EARLY_STOPPING_PATIENCE = 5
ROUTER_LORA_LR = 1e-4
ROUTER_HEAD_LR = 2e-4
ROUTER_WEIGHT_DECAY = 0.01
ROUTER_WARMUP_RATIO = 0.05
ROUTER_MAX_GRAD_NORM = 1.0
ROUTER_WARMUP_PROMPTS = 2

# --- Decision-aware loss ---
SAFETY_LOSS_WEIGHT = 1.0
LATENCY_LOSS_WEIGHT = 0.75
TOKEN_LOSS_WEIGHT = 0.10
OPPORTUNITY_MARGIN_LOSS_WEIGHT = 0.25
MISSED_OPPORTUNITY_WEIGHT = 2.0
UNSAFE_SELECTION_WEIGHT = 4.0
SAFETY_LOGIT_MARGIN = 1.0

# --- Validation-only calibration and selector search ---
PLATT_FOLDS = 5
SAFETY_THRESHOLD_GRID = (0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 0.975)
LATENCY_BLEND_GRID = (0.0, 0.25, 0.50, 0.75, 1.0)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert V4_SCOPE_SCHEMA_VERSION == 4
assert REQUIRED_V3_SCHEMA_VERSION == 3
assert N_PER_TASK == 300 and len(TASKS) == 3
assert MIN_QUALITY_RETENTION == 0.98
assert SAFETY_DEFINITION == "candidate_quality >= fallback_quality - epsilon"
assert ROUTER_LORA_R == 4 and ROUTER_LORA_ALPHA == 8
assert ROUTER_LORA_TARGET_MODULES == "all-linear"
assert ROUTER_MAX_INPUT_TOKENS == 512
assert MISSED_OPPORTUNITY_WEIGHT == 2.0
assert UNSAFE_SELECTION_WEIGHT == 4.0
assert PLATT_FOLDS == 5
assert MIN_PREDICTED_SPEEDUP == 0.02
assert torch.cuda.is_available(), "Choose a GPU runtime in Colab."
assert transformers.__version__ == "4.53.1", (
    f"Expected transformers 4.53.1, found {transformers.__version__}. "
    "Restart the runtime after the install cell."
)

from google.colab import drive
drive.mount("/content/drive")
V3_ROOT = Path("/content/drive/MyDrive/llm_router_v3")
V4_REPORTS = V3_ROOT / "reports_v4"
V4_ARTIFACTS = V3_ROOT / "artifacts_v4"
V4_REPORTS.mkdir(parents=True, exist_ok=True)
V4_ARTIFACTS.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"

v3_manifest_path = V3_ROOT / "run_manifest_v3.json"
if not v3_manifest_path.exists():
    raise FileNotFoundError(
        "Complete notebook 03 first; run_manifest_v3.json was not found."
    )
V3_MANIFEST = json.loads(v3_manifest_path.read_text(encoding="utf-8"))
EVIDENCE_FINGERPRINT = V3_MANIFEST["run_fingerprint"]
GPU_NAME = torch.cuda.get_device_name(0)
COMPUTE_DTYPE = (
    torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8
    else torch.float16
)
assert GPU_NAME == V3_MANIFEST["gpu"], (
    "Router overhead must be measured on the same GPU type as the reused "
    f"candidate latencies. V3 used {V3_MANIFEST['gpu']!r}; current GPU is "
    f"{GPU_NAME!r}."
)

V4_ROUTER_CONTRACT = {
    "scope_schema_version": V4_SCOPE_SCHEMA_VERSION,
    "scope_file": V4_SCOPE_FILE_NAME,
    "evidence_fingerprint": EVIDENCE_FINGERPRINT,
    "seed": SEED,
    "tasks": TASKS,
    "models": MODEL_NAMES,
    "split": [0.60, 0.20, 0.20],
    "safety_definition": SAFETY_DEFINITION,
    "quality_safety_epsilon": QUALITY_SAFETY_EPSILON,
    "minimum_quality_retention": MIN_QUALITY_RETENTION,
    "minimum_predicted_speedup": MIN_PREDICTED_SPEEDUP,
    "encoder_repo": ROUTER_ENCODER_REPO,
    "encoder_revision": ROUTER_ENCODER_REVISION,
    "router_max_input_tokens": ROUTER_MAX_INPUT_TOKENS,
    "lora": {
        "r": ROUTER_LORA_R,
        "alpha": ROUTER_LORA_ALPHA,
        "dropout": ROUTER_LORA_DROPOUT,
        "target_modules": ROUTER_LORA_TARGET_MODULES,
    },
    "optimization": {
        "batch_size": ROUTER_BATCH_SIZE,
        "gradient_accumulation": ROUTER_GRADIENT_ACCUMULATION,
        "max_epochs": ROUTER_MAX_EPOCHS,
        "min_epochs": ROUTER_MIN_EPOCHS,
        "early_stopping_patience": ROUTER_EARLY_STOPPING_PATIENCE,
        "lora_lr": ROUTER_LORA_LR,
        "head_lr": ROUTER_HEAD_LR,
        "weight_decay": ROUTER_WEIGHT_DECAY,
        "warmup_ratio": ROUTER_WARMUP_RATIO,
        "max_grad_norm": ROUTER_MAX_GRAD_NORM,
    },
    "loss": {
        "safety": SAFETY_LOSS_WEIGHT,
        "latency": LATENCY_LOSS_WEIGHT,
        "tokens": TOKEN_LOSS_WEIGHT,
        "opportunity_margin": OPPORTUNITY_MARGIN_LOSS_WEIGHT,
        "missed_opportunity_multiplier": MISSED_OPPORTUNITY_WEIGHT,
        "unsafe_selection_multiplier": UNSAFE_SELECTION_WEIGHT,
        "safety_logit_margin": SAFETY_LOGIT_MARGIN,
        "base_bce_pos_weight": 1.0,
    },
    "calibration": {"method": "per-candidate Platt", "folds": PLATT_FOLDS},
    "safety_threshold_grid": SAFETY_THRESHOLD_GRID,
    "latency_blend_grid": LATENCY_BLEND_GRID,
    "checkpoint_rule": "calibrated overhead-inclusive validation routing",
    "deployment_guard": "disable unless retention >= 0.98 and net savings > 0",
    "gpu": GPU_NAME,
    "dtype": str(COMPUTE_DTYPE),
    "packages": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "peft": peft.__version__,
        "sklearn": sklearn.__version__,
    },
}
ROUTER_FINGERPRINT = hashlib.sha256(
    json.dumps(V4_ROUTER_CONTRACT, sort_keys=True).encode()
).hexdigest()
(V4_REPORTS / "synchronized_contract_v4.json").write_text(
    json.dumps(V4_ROUTER_CONTRACT, indent=2), encoding="utf-8"
)
print({
    "gpu": GPU_NAME,
    "dtype": str(COMPUTE_DTYPE),
    "evidence_tag": EVIDENCE_FINGERPRINT[:16],
    "router_tag": ROUTER_FINGERPRINT[:16],
    "reports": str(V4_REPORTS),
    "artifacts": str(V4_ARTIFACTS),
})


Mounted at /content/drive
{'gpu': 'Tesla T4', 'dtype': 'torch.float16', 'evidence_tag': 'd2044a470f45f714', 'router_tag': '25c4fd832dd92177', 'reports': '/content/drive/MyDrive/llm_router_v3/reports_v4', 'artifacts': '/content/drive/MyDrive/llm_router_v3/artifacts_v4'}


## 2. Load and audit the immutable v3 evidence

Reuse is safe only when the data are exactly what v4 expects. This section
checks the v3 contract, model repositories, prompt counts, measurement
fingerprint, and rectangular prompt-model panel before any learning begins.

V4 intentionally reads `measurements_v3.parquet` rather than candidate cache
files individually. Notebook 03 already parsed and scored those responses;
v4 changes the routing target, not the benchmark evaluator.

**Expected output:** 900 prompts, 2,700 measurements, and the same candidate
summary seen at the end of v3's measurement stage.


In [3]:
assert V3_MANIFEST["schema_version"] == REQUIRED_V3_SCHEMA_VERSION
assert V3_MANIFEST["prompt_template_version"] == REQUIRED_V3_PROMPT_TEMPLATE
assert V3_MANIFEST["seed"] == SEED
assert int(V3_MANIFEST["n_per_task"]) == N_PER_TASK
assert tuple(V3_MANIFEST["tasks"]) == TASKS
assert set(V3_MANIFEST["models"]) == set(MODEL_NAMES)
for model_name, expected_repo in EXPECTED_MODEL_REPOS.items():
    assert V3_MANIFEST["models"][model_name]["repo"] == expected_repo

prompts_path = V3_ROOT / "data" / "prompts_v3.parquet"
measurements_path = V3_ROOT / "data" / "measurements_v3.parquet"
if not prompts_path.exists() or not measurements_path.exists():
    raise FileNotFoundError(
        "Notebook 03 must finish its scoring stage before v4 can run."
    )

prompts = pd.read_parquet(prompts_path)
measurements = pd.read_parquet(measurements_path)
prompt_columns = {
    "prompt_id", "task", "subject", "num_choices", "prompt_words",
    "length_bin", "prompt", "reference",
}
measurement_columns = {
    "prompt_id", "task", "model", "model_repo", "run_fingerprint",
    "generation_s", "output_tokens", "quality", "parsed", "strict_format",
}
assert not (prompt_columns - set(prompts.columns))
assert not (measurement_columns - set(measurements.columns))
assert prompts.prompt_id.is_unique
assert len(prompts) == N_PER_TASK * len(TASKS)
assert prompts.groupby("task").size().reindex(TASKS).eq(N_PER_TASK).all()

measurements = measurements.drop_duplicates(["prompt_id", "model"], keep="last")
assert set(measurements.model) == set(MODEL_NAMES)
assert measurements.run_fingerprint.eq(EVIDENCE_FINGERPRINT).all()
assert len(measurements) == len(prompts) * len(MODEL_NAMES)
panel_counts = measurements.groupby("prompt_id").model.nunique()
assert panel_counts.eq(len(MODEL_NAMES)).all()
assert measurements.generation_s.gt(0).all()
assert measurements.output_tokens.ge(0).all()
assert measurements.quality.isin([0.0, 1.0]).all()
assert np.isfinite(measurements[["generation_s", "output_tokens", "quality"]]).all().all()

repository_check = (
    measurements[["model", "model_repo"]].drop_duplicates()
    .set_index("model").model_repo.to_dict()
)
assert repository_check == EXPECTED_MODEL_REPOS

evidence_summary = measurements.groupby(["model", "task"]).agg(
    prompts=("prompt_id", "size"),
    accuracy=("quality", "mean"),
    mean_latency_s=("generation_s", "mean"),
    mean_output_tokens=("output_tokens", "mean"),
    parse_rate=("parsed", "mean"),
    strict_format_rate=("strict_format", "mean"),
)
display(evidence_summary.round(3))
print({
    "prompts": len(prompts),
    "measurements": len(measurements),
    "evidence_fingerprint": EVIDENCE_FINGERPRINT[:16],
})


prompts  accuracy  mean_latency_s  \
model             task                                               
fast-dllm-v2-1.5b arc_challenge      300     0.707           0.290   
                  gsm8k              300     0.140           1.485   
                  mmlu               300     0.490           0.341   
qwen2.5-1.5b-ar   arc_challenge      300     0.683           0.318   
                  gsm8k              300     0.087           0.323   
                  mmlu               300     0.507           0.327   
qwen2.5-7b-4bit   arc_challenge      300     0.873           0.577   
                  gsm8k              300     0.203           0.629   
                  mmlu               300     0.680           0.629   

                                 mean_output_tokens  parse_rate  \
model             task                                            
fast-dllm-v2-1.5b arc_challenge              10.657       0.987   
                  gsm8k                      80.050       0.970   
                  mmlu                       11.217       0.993   
qwen2.5-1.5b-ar   arc_challenge               8.450       1.000   
                  gsm8k                       8.437       0.990   
                  mmlu                        8.850       1.000   
qwen2.5-7b-4bit   arc_challenge               7.000       1.000   
                  gsm8k                       8.257       1.000   
                  mmlu                        7.007       1.000   

                                 strict_format_rate  
model             task                               
fast-dllm-v2-1.5b arc_challenge               0.930  
                  gsm8k                       0.447  
                  mmlu                        0.903  
qwen2.5-1.5b-ar   arc_challenge               0.667  
                  gsm8k                       1.000  
                  mmlu                        0.590  
qwen2.5-7b-4bit   arc_challenge               1.000  
                  gsm8k                       1.000  
                  mmlu                        1.000

{'prompts': 900, 'measurements': 2700, 'evidence_fingerprint': 'd2044a470f45f714'}


## 3. Split prompts and construct decision-aligned targets

The split occurs at the prompt level, never at the prompt-model row level.
The fallback is the model with the highest **training-only** mean accuracy.

For each alternative, v4 asks whether its observed quality is at least the
fallback's quality. A safe alternative receives extra training emphasis only
when it also offers measured latency savings. An unsafe alternative receives
a stronger penalty when the fallback was correct and it was wrong.

Training-only task/model median latencies provide a stable baseline for the
later validation-selected latency blend.


In [4]:
def wide(column: str) -> pd.DataFrame:
    return measurements.pivot(
        index="prompt_id", columns="model", values=column
    ).reindex(columns=MODEL_NAMES)

table = prompts.set_index("prompt_id").join(wide("quality").add_prefix("q__"))
table = table.join(wide("generation_s").add_prefix("l__"))
table = table.join(wide("output_tokens").add_prefix("t__")).reset_index()

train_val, test = train_test_split(
    table.index, test_size=0.20, random_state=SEED, stratify=table.task
)
train, validation = train_test_split(
    train_val, test_size=0.25, random_state=SEED + 1,
    stratify=table.loc[train_val, "task"],
)
masks = {
    "train": table.index.isin(train),
    "validation": table.index.isin(validation),
    "test": table.index.isin(test),
}
display(pd.DataFrame({
    name: table.loc[mask].groupby("task").size()
    for name, mask in masks.items()
}).fillna(0).astype(int))

Q = table[[f"q__{name}" for name in MODEL_NAMES]].to_numpy(float)
L = table[[f"l__{name}" for name in MODEL_NAMES]].to_numpy(float)
T = table[[f"t__{name}" for name in MODEL_NAMES]].to_numpy(float)
strongest_idx = int(Q[masks["train"]].mean(axis=0).argmax())
fallback_name = MODEL_NAMES[strongest_idx]
nonfallback_indices = np.array([
    index for index in range(len(MODEL_NAMES)) if index != strongest_idx
])
NONFALLBACK_NAMES = tuple(MODEL_NAMES[index] for index in nonfallback_indices)

fallback_quality = Q[:, strongest_idx][:, None]
fallback_latency = L[:, strongest_idx][:, None]
alternative_quality = Q[:, nonfallback_indices]
alternative_latency = L[:, nonfallback_indices]
replacement_safe = (
    alternative_quality >= fallback_quality - QUALITY_SAFETY_EPSILON
)
normalized_gain = np.maximum(
    0.0, (fallback_latency - alternative_latency) / np.maximum(fallback_latency, 1e-9)
)
opportunity_gain = replacement_safe.astype(float) * normalized_gain
quality_drop = np.maximum(0.0, fallback_quality - alternative_quality)
safety_example_weight = np.where(
    replacement_safe,
    1.0 + MISSED_OPPORTUNITY_WEIGHT * opportunity_gain,
    1.0 + UNSAFE_SELECTION_WEIGHT * quality_drop,
)

# Every row receives the median for its task/model, estimated from train only.
train_long = measurements.loc[
    measurements.prompt_id.isin(set(table.loc[masks["train"], "prompt_id"]))
]
task_model_median = train_long.pivot_table(
    index="task", columns="model", values="generation_s", aggfunc="median"
).reindex(index=TASKS, columns=MODEL_NAMES)
global_model_median = np.median(L[masks["train"]], axis=0)
task_latency_baseline = np.vstack([
    task_model_median.loc[task].fillna(
        pd.Series(global_model_median, index=MODEL_NAMES)
    ).to_numpy(float)
    for task in table.task
])

text = (
    "[TASK=" + table.task.astype(str) + "] "
    + "[SUBJECT=" + table.subject.astype(str) + "] "
    + "[CHOICES=" + table.num_choices.astype(str) + "] "
    + "[LENGTH_BIN=" + table.length_bin.astype(str) + "] "
    + table.prompt.astype(str)
).to_numpy()

fastest_idx = int(L[masks["train"]].mean(axis=0).argmin())
actual_replacement_eligible = Q >= Q[:, strongest_idx][:, None]
oracle_idx = np.where(actual_replacement_eligible, L, np.inf).argmin(axis=1)

print({
    "fallback": fallback_name,
    "nonfallback_candidates": NONFALLBACK_NAMES,
    "always_fastest_baseline": MODEL_NAMES[fastest_idx],
    "train_safe_rates": dict(zip(
        NONFALLBACK_NAMES,
        replacement_safe[masks["train"]].mean(axis=0).round(3),
    )),
    "train_safe_faster_rates": dict(zip(
        NONFALLBACK_NAMES,
        (opportunity_gain[masks["train"]] > 0).mean(axis=0).round(3),
    )),
})
display(task_model_median.round(3))


,train,validation,test
task,,,
arc_challenge,180,60,60
gsm8k,180,60,60
mmlu,180,60,60


{'fallback': 'qwen2.5-7b-4bit', 'nonfallback_candidates': ('qwen2.5-1.5b-ar', 'fast-dllm-v2-1.5b'), 'always_fastest_baseline': 'qwen2.5-1.5b-ar', 'train_safe_rates': {'qwen2.5-1.5b-ar': np.float64(0.819), 'fast-dllm-v2-1.5b': np.float64(0.809)}, 'train_safe_faster_rates': {'qwen2.5-1.5b-ar': np.float64(0.804), 'fast-dllm-v2-1.5b': np.float64(0.67)}}


model,qwen2.5-1.5b-ar,fast-dllm-v2-1.5b,qwen2.5-7b-4bit
task,,,
gsm8k,0.286,0.474,0.624
mmlu,0.337,0.288,0.606
arc_challenge,0.277,0.276,0.593


## 4. Smaller ModernBERT router and decision-aware loss

Rank-4 LoRA reduces trainable encoder capacity relative to v3. The fallback
remains permanently available, so the safety head predicts only the two
possible replacements. Latency and output-token heads still predict all
three candidates.

The base BCE is deliberately unweighted (`pos_weight=1`). Per-example
multipliers then encode actual decision cost:

- missing a safe speedup is weighted by its latency opportunity;
- claiming an unsafe replacement is safe receives a larger quality penalty;
- a margin term pushes genuinely safe, faster candidates toward confident
  positive logits.

Uniform shuffling preserves the empirical prompt distribution.


In [5]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

safety_targets = torch.tensor(replacement_safe.astype(float), dtype=torch.float32)
safety_weights = torch.tensor(safety_example_weight, dtype=torch.float32)
opportunity_targets = torch.tensor(opportunity_gain, dtype=torch.float32)
latency_targets = torch.tensor(np.log1p(L), dtype=torch.float32)
token_targets = torch.tensor(np.log1p(T), dtype=torch.float32)

router_tokenizer = AutoTokenizer.from_pretrained(
    ROUTER_ENCODER_REPO, revision=ROUTER_ENCODER_REVISION
)

class DecisionAlignedRouter(torch.nn.Module):
    def __init__(self):
        super().__init__()
        base_encoder = AutoModel.from_pretrained(
            ROUTER_ENCODER_REPO,
            revision=ROUTER_ENCODER_REVISION,
            attn_implementation="sdpa",
        )
        lora_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            inference_mode=False,
            r=ROUTER_LORA_R,
            lora_alpha=ROUTER_LORA_ALPHA,
            lora_dropout=ROUTER_LORA_DROPOUT,
            target_modules=ROUTER_LORA_TARGET_MODULES,
            bias="none",
        )
        self.encoder = get_peft_model(base_encoder, lora_config)
        hidden_size = int(base_encoder.config.hidden_size)
        self.dropout = torch.nn.Dropout(0.10)
        self.safety_head = torch.nn.Linear(hidden_size, len(NONFALLBACK_NAMES))
        self.latency_head = torch.nn.Linear(hidden_size, len(MODEL_NAMES))
        self.token_head = torch.nn.Linear(hidden_size, len(MODEL_NAMES))

    def forward(self, **inputs):
        hidden = self.encoder(**inputs).last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
        pooled = self.dropout(pooled)
        return {
            "safety_logits": self.safety_head(pooled),
            "latency_log": self.latency_head(pooled),
            "token_log": self.token_head(pooled),
        }

router_load_started = time.perf_counter()
router_model = DecisionAlignedRouter().to("cuda")
router_load_time_s = time.perf_counter() - router_load_started
router_model.encoder.print_trainable_parameters()

def encode_indices(indices):
    indices = [int(index) for index in indices]
    encoded = router_tokenizer(
        [f"classification: {text[index]}" for index in indices],
        padding=True,
        truncation=True,
        max_length=ROUTER_MAX_INPUT_TOKENS,
        return_tensors="pt",
    )
    return torch.tensor(indices, dtype=torch.long), encoded

train_indices = np.flatnonzero(masks["train"]).tolist()
validation_indices = np.flatnonzero(masks["validation"])
test_indices = np.flatnonzero(masks["test"])
train_loader = DataLoader(
    train_indices,
    batch_size=ROUTER_BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    collate_fn=encode_indices,
)

def router_batch_loss(outputs, indices):
    s_target = safety_targets[indices].to("cuda")
    s_weight = safety_weights[indices].to("cuda")
    opportunity = opportunity_targets[indices].to("cuda")
    l_target = latency_targets[indices].to("cuda")
    t_target = token_targets[indices].to("cuda")

    # No prevalence-based pos_weight: positives and negatives begin equally.
    element_bce = F.binary_cross_entropy_with_logits(
        outputs["safety_logits"].float(), s_target, reduction="none"
    )
    safety_loss = (element_bce * s_weight).mean()
    latency_loss = F.smooth_l1_loss(outputs["latency_log"].float(), l_target)
    token_loss = F.smooth_l1_loss(outputs["token_log"].float(), t_target)
    opportunity_margin_loss = (
        F.softplus(SAFETY_LOGIT_MARGIN - outputs["safety_logits"].float())
        * opportunity
    ).mean()
    total = (
        SAFETY_LOSS_WEIGHT * safety_loss
        + LATENCY_LOSS_WEIGHT * latency_loss
        + TOKEN_LOSS_WEIGHT * token_loss
        + OPPORTUNITY_MARGIN_LOSS_WEIGHT * opportunity_margin_loss
    )
    return (
        total, safety_loss, latency_loss, token_loss,
        opportunity_margin_loss,
    )

@torch.inference_mode()
def evaluate_loss(indices, batch_size=ROUTER_BATCH_SIZE * 2):
    router_model.eval()
    loader = DataLoader(
        [int(index) for index in indices], batch_size=batch_size,
        shuffle=False, collate_fn=encode_indices,
    )
    totals = np.zeros(5, dtype=float)
    examples = 0
    for batch_indices, encoded in loader:
        encoded = encoded.to("cuda")
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
            losses = router_batch_loss(router_model(**encoded), batch_indices)
        examples += len(batch_indices)
        totals += len(batch_indices) * np.array([
            float(loss.detach()) for loss in losses
        ])
    return totals / examples

@torch.inference_mode()
def predict_indices(indices, measure_batch_one_overhead=False):
    """Return predictions in the same order as indices.

    When overhead measurement is requested, prompt tokenization and the GPU
    forward pass are both timed, matching the online decision boundary.
    """
    router_model.eval()
    indices = np.asarray(indices, dtype=int)
    safety_parts, latency_parts, token_parts = [], [], []
    overhead_s = np.zeros(len(indices), dtype=float)

    if measure_batch_one_overhead:
        warmup = indices[:ROUTER_WARMUP_PROMPTS]
        for index in warmup:
            _, encoded = encode_indices([int(index)])
            with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
                router_model(**encoded.to("cuda"))
        torch.cuda.synchronize()

        for position, index in enumerate(indices):
            torch.cuda.synchronize()
            started = time.perf_counter()
            _, encoded = encode_indices([int(index)])
            with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
                outputs = router_model(**encoded.to("cuda"))
            torch.cuda.synchronize()
            overhead_s[position] = time.perf_counter() - started
            safety_parts.append(outputs["safety_logits"].float().cpu().numpy())
            latency_parts.append(outputs["latency_log"].float().cpu().numpy())
            token_parts.append(outputs["token_log"].float().cpu().numpy())
    else:
        loader = DataLoader(
            indices.tolist(), batch_size=ROUTER_BATCH_SIZE * 2,
            shuffle=False, collate_fn=encode_indices,
        )
        for _, encoded in loader:
            with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
                outputs = router_model(**encoded.to("cuda"))
            safety_parts.append(outputs["safety_logits"].float().cpu().numpy())
            latency_parts.append(outputs["latency_log"].float().cpu().numpy())
            token_parts.append(outputs["token_log"].float().cpu().numpy())

    return {
        "indices": indices,
        "safety_logits": np.concatenate(safety_parts),
        "latency": np.maximum(1e-6, np.expm1(np.concatenate(latency_parts))),
        "tokens": np.maximum(1.0, np.expm1(np.concatenate(token_parts))),
        "overhead_s": overhead_s,
    }


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/596M [00:00<?, ?B/s]

trainable params: 844,800 || all params: 149,859,072 || trainable%: 0.5637


## 5. Platt calibration and the exact deployment selector

Weighted or asymmetric training losses intentionally distort raw
probabilities. Platt scaling repairs both their slope and intercept:

```text
calibrated probability = sigmoid(slope * raw logit + intercept)
```

During checkpoint comparison, five-fold out-of-fold calibration ensures a
validation row is never calibrated by a model fitted on its own label.

The selector combines neural latency with a training-only task/model median.
Validation chooses the blend and one threshold per replacement candidate.
A candidate must also have at least 2% predicted speedup. The fallback is
always eligible.


In [7]:
def fit_platt(logits, targets):
    logits = np.asarray(logits, dtype=np.float32)
    targets = np.asarray(targets, dtype=np.float32)
    prevalence = float(targets.mean())
    if prevalence <= 0.0 or prevalence >= 1.0:
        clipped = np.clip(prevalence, 1e-4, 1 - 1e-4)
        return 0.0, float(np.log(clipped / (1 - clipped)))

    x = torch.tensor(logits, dtype=torch.float32)
    y = torch.tensor(targets, dtype=torch.float32)
    raw_slope = torch.nn.Parameter(torch.tensor(0.5413249))  # softplus -> 1
    intercept = torch.nn.Parameter(torch.tensor(0.0))
    optimizer = torch.optim.LBFGS(
        [raw_slope, intercept], lr=0.1, max_iter=100,
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad()
        slope = F.softplus(raw_slope) + 1e-4
        calibrated_logits = slope * x + intercept
        loss = F.binary_cross_entropy_with_logits(calibrated_logits, y)
        loss.backward()
        return loss

    optimizer.step(closure)
    return (
        float((F.softplus(raw_slope) + 1e-4).detach()),
        float(intercept.detach()),
    )

def apply_platt(logits, parameters):
    probabilities = np.zeros_like(logits, dtype=float)
    for column, (slope, intercept) in enumerate(parameters):
        calibrated_logit = np.clip(
            slope * logits[:, column] + intercept, -40, 40
        )
        probabilities[:, column] = 1 / (1 + np.exp(-calibrated_logit))
    return probabilities

def out_of_fold_platt(logits, targets, seed=SEED):
    probabilities = np.zeros_like(logits, dtype=float)
    parameters_by_candidate = {}
    for column, candidate in enumerate(NONFALLBACK_NAMES):
        candidate_target = targets[:, column].astype(int)
        splitter = StratifiedKFold(
            n_splits=PLATT_FOLDS, shuffle=True,
            random_state=seed + column,
        )
        fold_parameters = []
        for fit_rows, heldout_rows in splitter.split(logits, candidate_target):
            parameters = fit_platt(
                logits[fit_rows, column], candidate_target[fit_rows]
            )
            fold_parameters.append(parameters)
            probabilities[heldout_rows, column] = apply_platt(
                logits[heldout_rows, column, None], [parameters]
            )[:, 0]
        parameters_by_candidate[candidate] = fold_parameters
    return probabilities, parameters_by_candidate

def blended_latency(neural_latency, indices, blend):
    return (
        blend * neural_latency
        + (1.0 - blend) * task_latency_baseline[np.asarray(indices)]
    )

def select_routes(safety_probability, latency_prediction, thresholds):
    eligible = np.zeros_like(latency_prediction, dtype=bool)
    eligible[:, strongest_idx] = True
    fallback_prediction = latency_prediction[:, strongest_idx]
    for column, model_index in enumerate(nonfallback_indices):
        safe_enough = safety_probability[:, column] >= thresholds[column]
        fast_enough = (
            latency_prediction[:, model_index]
            <= fallback_prediction * (1.0 - MIN_PREDICTED_SPEEDUP)
        )
        eligible[:, model_index] = safe_enough & fast_enough
    chosen = np.where(eligible, latency_prediction, np.inf).argmin(axis=1)
    selected_fallback = chosen == strongest_idx
    no_eligible_alternative = ~eligible[:, nonfallback_indices].any(axis=1)
    return chosen, eligible, selected_fallback, no_eligible_alternative

def route_metrics(indices, chosen, overhead_s=None):
    indices = np.asarray(indices, dtype=int)
    chosen = np.asarray(chosen, dtype=int)
    row = np.arange(len(indices))
    chosen_q = Q[indices][row, chosen]
    chosen_generation = L[indices][row, chosen]
    fallback_q = Q[indices, strongest_idx]
    fallback_generation = L[indices, strongest_idx]
    overhead = (
        np.zeros(len(indices), dtype=float)
        if overhead_s is None else np.asarray(overhead_s, dtype=float)
    )
    chosen_latency = chosen_generation + overhead
    actual_safe_faster = (
        (Q[indices] >= fallback_q[:, None] - QUALITY_SAFETY_EPSILON)
        & (L[indices] < fallback_generation[:, None])
    )
    missed_safe_opportunity = (
        actual_safe_faster.any(axis=1)
        & (chosen_generation >= fallback_generation)
    )
    return {
        "accuracy": float(chosen_q.mean()),
        "fallback_accuracy": float(fallback_q.mean()),
        "accuracy_delta": float(chosen_q.mean() - fallback_q.mean()),
        "quality_retention": float(
            chosen_q.mean() / max(fallback_q.mean(), 1e-9)
        ),
        "quality_loss_rate": float(np.mean(chosen_q < fallback_q)),
        "mean_quality_regret": float(
            np.maximum(0.0, fallback_q - chosen_q).mean()
        ),
        "p95_quality_regret": float(
            np.quantile(
                np.maximum(0.0, fallback_q - chosen_q),
                0.95, method="higher",
            )
        ),
        "generation_s": float(chosen_generation.mean()),
        "router_overhead_s": float(overhead.mean()),
        "latency_s": float(chosen_latency.mean()),
        "latency_reduction": float(
            1.0 - chosen_latency.mean() / fallback_generation.mean()
        ),
        "fallback_usage": float(np.mean(chosen == strongest_idx)),
        "missed_safe_opportunity_rate": float(missed_safe_opportunity.mean()),
    }

def search_selector(prediction, calibrated_probability):
    indices = prediction["indices"]
    rows = []
    for blend in LATENCY_BLEND_GRID:
        latency_prediction = blended_latency(
            prediction["latency"], indices, blend
        )
        for thresholds in itertools.product(
            SAFETY_THRESHOLD_GRID, repeat=len(NONFALLBACK_NAMES)
        ):
            chosen, eligible, selected_fallback, no_eligible = select_routes(
                calibrated_probability, latency_prediction,
                np.asarray(thresholds),
            )
            metrics = route_metrics(
                indices, chosen, prediction["overhead_s"]
            )
            rows.append({
                "latency_blend": blend,
                **{
                    f"threshold__{candidate}": threshold
                    for candidate, threshold in zip(NONFALLBACK_NAMES, thresholds)
                },
                **metrics,
                "eligible_alternative_rate": float(
                    eligible[:, nonfallback_indices].any(axis=1).mean()
                ),
                "no_eligible_alternative_rate": float(no_eligible.mean()),
            })
    search = pd.DataFrame(rows)
    feasible = search.loc[
        search.quality_retention.ge(MIN_QUALITY_RETENTION)
        & search.latency_reduction.gt(0)
    ]
    if feasible.empty:
        return search, None
    threshold_columns = [
        f"threshold__{candidate}" for candidate in NONFALLBACK_NAMES
    ]
    best = feasible.assign(
        threshold_sum=feasible[threshold_columns].sum(axis=1)
    ).sort_values(
        ["latency_reduction", "quality_loss_rate", "threshold_sum"],
        ascending=[False, True, False],
    ).iloc[0]
    return search, best


## 6. Train and choose checkpoints by deployed behavior

Generic validation loss remains diagnostic. The primary checkpoint score is
the actual downstream decision:

1. measure validation inference at batch size one;
2. calibrate safety out of fold;
3. search thresholds and latency blends;
4. include each prompt's measured router overhead; and
5. require at least 98% quality retention and positive net savings.

A feasible checkpoint always outranks an infeasible one. Among feasible
checkpoints, net latency reduction is the primary objective. Early stopping
restores only trainable LoRA and head parameters.


In [8]:
encoder_parameters = [
    parameter for parameter in router_model.encoder.parameters()
    if parameter.requires_grad
]
head_parameters = [
    parameter for name, parameter in router_model.named_parameters()
    if not name.startswith("encoder.")
]
parameter_groups = [
    {"params": encoder_parameters, "lr": ROUTER_LORA_LR},
    {"params": head_parameters, "lr": ROUTER_HEAD_LR},
]
optimizer = torch.optim.AdamW(
    parameter_groups, weight_decay=ROUTER_WEIGHT_DECAY
)
updates_per_epoch = math.ceil(
    len(train_loader) / ROUTER_GRADIENT_ACCUMULATION
)
total_training_steps = updates_per_epoch * ROUTER_MAX_EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(
        1, round(total_training_steps * ROUTER_WARMUP_RATIO)
    ),
    num_training_steps=total_training_steps,
)
scaler = torch.amp.GradScaler(
    "cuda", enabled=COMPUTE_DTYPE == torch.float16
)

trainable_names = {
    name for name, parameter in router_model.named_parameters()
    if parameter.requires_grad
}
history = []
best_key = (-1, -np.inf, -np.inf)
best_state = None
best_epoch = 0
epochs_without_improvement = 0
training_started = time.perf_counter()

for epoch in range(1, ROUTER_MAX_EPOCHS + 1):
    router_model.train()
    optimizer.zero_grad(set_to_none=True)
    train_totals = np.zeros(5, dtype=float)
    train_examples = 0
    accumulation_count = 0

    for step, (batch_indices, encoded) in enumerate(train_loader, 1):
        encoded = encoded.to("cuda")
        accumulation_count += 1
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
            losses = router_batch_loss(
                router_model(**encoded), batch_indices
            )
            scaled_loss = losses[0] / ROUTER_GRADIENT_ACCUMULATION
        scaler.scale(scaled_loss).backward()
        should_step = (
            accumulation_count == ROUTER_GRADIENT_ACCUMULATION
            or step == len(train_loader)
        )
        if should_step:
            if accumulation_count != ROUTER_GRADIENT_ACCUMULATION:
                correction = (
                    ROUTER_GRADIENT_ACCUMULATION / accumulation_count
                )
                for parameter in router_model.parameters():
                    if parameter.grad is not None:
                        parameter.grad.mul_(correction)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [
                    parameter for parameter in router_model.parameters()
                    if parameter.requires_grad
                ],
                ROUTER_MAX_GRAD_NORM,
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            accumulation_count = 0

        batch_size = len(batch_indices)
        train_examples += batch_size
        train_totals += batch_size * np.array([
            float(loss.detach()) for loss in losses
        ])

    validation_losses = evaluate_loss(validation_indices)
    validation_prediction = predict_indices(
        validation_indices, measure_batch_one_overhead=True
    )
    validation_targets = replacement_safe[validation_indices]
    oof_probability, _ = out_of_fold_platt(
        validation_prediction["safety_logits"], validation_targets,
        seed=SEED,
    )
    epoch_search, epoch_best = search_selector(
        validation_prediction, oof_probability
    )
    feasible_checkpoint = epoch_best is not None
    if feasible_checkpoint:
        route_score = float(epoch_best.latency_reduction)
        quality_score = -float(epoch_best.quality_loss_rate)
    else:
        route_score = -float(validation_losses[0])
        quality_score = -float(validation_losses[1])

    candidate_key = (
        int(feasible_checkpoint), route_score, quality_score
    )
    record = {
        "epoch": epoch,
        "train_loss": train_totals[0] / train_examples,
        "validation_loss": validation_losses[0],
        "validation_safety_loss": validation_losses[1],
        "validation_latency_loss": validation_losses[2],
        "validation_token_loss": validation_losses[3],
        "validation_opportunity_margin_loss": validation_losses[4],
        "validation_route_feasible": feasible_checkpoint,
        "validation_route_score": route_score,
        "validation_quality_retention": (
            np.nan if epoch_best is None
            else float(epoch_best.quality_retention)
        ),
        "validation_fallback_usage": (
            np.nan if epoch_best is None
            else float(epoch_best.fallback_usage)
        ),
        "mean_router_overhead_ms": float(
            1000 * validation_prediction["overhead_s"].mean()
        ),
    }
    history.append(record)
    print({
        key: round(value, 4)
        if isinstance(value, (float, np.floating)) and np.isfinite(value)
        else value
        for key, value in record.items()
    })

    if candidate_key > best_key:
        best_key = candidate_key
        best_epoch = epoch
        best_state = {
            name: value.detach().cpu().clone()
            for name, value in router_model.state_dict().items()
            if name in trainable_names
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if (
            epoch >= ROUTER_MIN_EPOCHS
            and epochs_without_improvement
            >= ROUTER_EARLY_STOPPING_PATIENCE
        ):
            print(
                f"Early stopping after epoch {epoch}; "
                f"restoring epoch {best_epoch}."
            )
            break

router_training_time_s = time.perf_counter() - training_started
assert best_state is not None
router_model.load_state_dict(best_state, strict=False)
training_history = pd.DataFrame(history)
training_history.to_csv(
    V4_REPORTS / "training_history_v4.csv", index=False
)
display(training_history.round(4))
print({
    "best_epoch": best_epoch,
    "training_time_s": round(router_training_time_s, 3),
    "trainable_parameters": sum(
        parameter.numel() for parameter in router_model.parameters()
        if parameter.requires_grad
    ),
})


W0810 20:09:42.542000 1474 torch/_inductor/utils.py:1731] [1/0_1] Not enough SMs to use max_autotune_gemm mode
/tmp/ipykernel_1474/3497791243.py:80: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


{'epoch': 1, 'train_loss': np.float64(2.0752), 'validation_loss': np.float64(2.2126), 'validation_safety_loss': np.float64(2.0927), 'validation_latency_loss': np.float64(0.0554), 'validation_token_loss': np.float64(0.2614), 'validation_opportunity_margin_loss': np.float64(0.2088), 'validation_route_feasible': False, 'validation_route_score': -2.2126, 'validation_quality_retention': nan, 'validation_fallback_usage': nan, 'mean_router_overhead_ms': 48.4649}
{'epoch': 2, 'train_loss': np.float64(1.7882), 'validation_loss': np.float64(1.9673), 'validation_safety_loss': np.float64(1.8427), 'validation_latency_loss': np.float64(0.0394), 'validation_token_loss': np.float64(0.1702), 'validation_opportunity_margin_loss': np.float64(0.3123), 'validation_route_feasible': False, 'validation_route_score': -1.9673, 'validation_quality_retention': nan, 'validation_fallback_usage': nan, 'mean_router_overhead_ms': 51.9341}
{'epoch': 3, 'train_loss': np.float64(1.7797), 'validation_loss': np.float64(2.0

,epoch,train_loss,validation_loss,validation_safety_loss,validation_latency_loss,validation_token_loss,validation_opportunity_margin_loss,validation_route_feasible,validation_route_score,validation_quality_retention,validation_fallback_usage,mean_router_overhead_ms
0,1,2.0752,2.2126,2.0927,0.0554,0.2614,0.2088,False,-2.2126,NaN,NaN,48.4649
1,2,1.7882,1.9673,1.8427,0.0394,0.1702,0.3123,False,-1.9673,NaN,NaN,51.9341
2,3,1.7797,2.0361,1.9308,0.0349,0.1524,0.2556,False,-2.0361,NaN,NaN,51.3391
3,4,1.7387,1.9418,1.8243,0.0313,0.1495,0.3163,False,-1.9418,NaN,NaN,51.1669
4,5,1.7196,1.9547,1.8381,0.0301,0.1462,0.3178,False,-1.9547,NaN,NaN,45.8110
5,6,1.7092,1.9633,1.8532,0.0296,0.1462,0.2933,False,-1.9633,NaN,NaN,69.4804
6,7,1.6878,1.9618,1.8538,0.0291,0.1437,0.2875,False,-1.9618,NaN,NaN,47.6673
7,8,1.6861,1.9491,1.8384,0.0284,0.1445,0.2995,False,-1.9491,NaN,NaN,53.3969
8,9,1.6523,1.9837,1.8787,0.0287,0.1431,0.2768,False,-1.9837,NaN,NaN,65.1466


{'best_epoch': 4, 'training_time_s': 289.185, 'trainable_parameters': 850952}


## 7. Freeze validation choices, then open the sealed test

The restored checkpoint is evaluated on validation one final time. Out-of-
fold probabilities select thresholds and latency blending. Then one Platt
model per candidate is fitted on all validation rows for use on unseen test
prompts.

Only after those choices are frozen does test inference run. If validation
has no feasible setting, test deployment is the fallback with zero router
overhead. Diagnostic router predictions are still retained so failure modes
can be studied without pretending the router was deployed.


In [9]:
final_validation_prediction = predict_indices(
    validation_indices, measure_batch_one_overhead=True
)
final_validation_targets = replacement_safe[validation_indices]
final_oof_probability, validation_fold_parameters = out_of_fold_platt(
    final_validation_prediction["safety_logits"],
    final_validation_targets,
    seed=SEED,
)
selector_search, selected_validation_row = search_selector(
    final_validation_prediction, final_oof_probability
)
selector_search.to_csv(
    V4_REPORTS / "selector_search_v4.csv", index=False
)

final_platt_parameters = [
    fit_platt(
        final_validation_prediction["safety_logits"][:, column],
        final_validation_targets[:, column],
    )
    for column in range(len(NONFALLBACK_NAMES))
]

if selected_validation_row is None:
    warnings.warn(
        "No validation configuration met 98% retention with positive net "
        "latency reduction. The deployed policy is fallback-only."
    )
    router_active = False
    selected_thresholds = np.ones(len(NONFALLBACK_NAMES), dtype=float)
    selected_latency_blend = 0.0
else:
    router_active = True
    selected_thresholds = np.array([
        float(selected_validation_row[f"threshold__{candidate}"])
        for candidate in NONFALLBACK_NAMES
    ])
    selected_latency_blend = float(
        selected_validation_row.latency_blend
    )

test_prediction = predict_indices(
    test_indices, measure_batch_one_overhead=True
)
test_probability = apply_platt(
    test_prediction["safety_logits"], final_platt_parameters
)
test_latency_prediction = blended_latency(
    test_prediction["latency"], test_indices, selected_latency_blend
)
diagnostic_test_idx, diagnostic_eligible, diagnostic_selected_fallback, diagnostic_no_eligible = (
    select_routes(
        test_probability, test_latency_prediction, selected_thresholds
    )
)

if router_active:
    router_test_idx = diagnostic_test_idx
    deployed_test_overhead = test_prediction["overhead_s"]
else:
    router_test_idx = np.full(len(test_indices), strongest_idx)
    deployed_test_overhead = np.zeros(len(test_indices), dtype=float)

strategy_choices = {
    "strongest": np.full(len(test_indices), strongest_idx),
    "fastest": np.full(len(test_indices), fastest_idx),
    "fallback_relative_oracle": oracle_idx[test_indices],
    "router": router_test_idx,
}
evaluation_rows = {}
for strategy, choices in strategy_choices.items():
    overhead = (
        deployed_test_overhead if strategy == "router" else None
    )
    evaluation_rows[strategy] = route_metrics(
        test_indices, choices, overhead
    )
evaluation = pd.DataFrame(evaluation_rows).T
evaluation.to_csv(V4_REPORTS / "evaluation_v4.csv")

display(selector_search.sort_values(
    ["quality_retention", "latency_reduction"], ascending=False
).head(12).round(4))
display(evaluation.round(4))
print({
    "router_active": router_active,
    "selected_thresholds": dict(zip(
        NONFALLBACK_NAMES, selected_thresholds.round(4)
    )),
    "selected_latency_blend": selected_latency_blend,
    "platt_parameters": {
        candidate: {"slope": round(parameters[0], 4),
                    "intercept": round(parameters[1], 4)}
        for candidate, parameters in zip(
            NONFALLBACK_NAMES, final_platt_parameters
        )
    },
    "mean_diagnostic_router_overhead_ms": round(
        1000 * test_prediction["overhead_s"].mean(), 3
    ),
})


/tmp/ipykernel_1474/811936009.py:26: UserWarning: No validation configuration met 98% retention with positive net latency reduction. The deployed policy is fallback-only.
  warnings.warn(


,latency_blend,threshold__qwen2.5-1.5b-ar,threshold__fast-dllm-v2-1.5b,accuracy,fallback_accuracy,accuracy_delta,quality_retention,quality_loss_rate,mean_quality_regret,p95_quality_regret,generation_s,router_overhead_s,latency_s,latency_reduction,fallback_usage,missed_safe_opportunity_rate,eligible_alternative_rate,no_eligible_alternative_rate
376,1.00,0.850,0.950,0.6,0.6,0.0,1.0,0.0,0.0,0.0,0.620,0.0576,0.6775,-0.0876,0.9889,0.8056,0.0111,0.9889
377,1.00,0.850,0.975,0.6,0.6,0.0,1.0,0.0,0.0,0.0,0.620,0.0576,0.6775,-0.0876,0.9889,0.8056,0.0111,0.9889
61,0.00,0.900,0.950,0.6,0.6,0.0,1.0,0.0,0.0,0.0,0.623,0.0576,0.6806,-0.0924,1.0000,0.8167,0.0000,1.0000
62,0.00,0.900,0.975,0.6,0.6,0.0,1.0,0.0,0.0,0.0,0.623,0.0576,0.6806,-0.0924,1.0000,0.8167,0.0000,1.0000
70,0.00,0.950,0.950,0.6,0.6,0.0,1.0,0.0,0.0,0.0,0.623,0.0576,0.6806,-0.0924,1.0000,0.8167,0.0000,1.0000
71,0.00,0.950,0.975,0.6,0.6,0.0,1.0,0.0,0.0,0.0,0.623,0.0576,0.6806,-0.0924,1.0000,0.8167,0.0000,1.0000
79,0.00,0.975,0.950,0.6,0.6,0.0,1.0,0.0,0.0,0.0,0.623,0.0576,0.6806,-0.0924,1.0000,0.8167,0.0000,1.0000
80,0.00,0.975,0.975,0.6,0.6,0.0,1.0,0.0,0.0,0.0,0.623,0.0576,0.6806,-0.0924,1.0000,0.8167,0.0000,1.0000
142,0.25,0.900,0.950,0.6,0.6,0.0,1.0,0.0,0.0,0.0,0.623,0.0576,0.6806,-0.0924,1.0000,0.8167,0.0000,1.0000
143,0.25,0.900,0.975,0.6,0.6,0.0,1.0,0.0,0.0,0.0,0.623,0.0576,0.6806,-0.0924,1.0000,0.8167,0.0000,1.0000


,accuracy,fallback_accuracy,accuracy_delta,quality_retention,quality_loss_rate,mean_quality_regret,p95_quality_regret,generation_s,router_overhead_s,latency_s,latency_reduction,fallback_usage,missed_safe_opportunity_rate
strongest,0.6389,0.6389,0.0000,1.0000,0.0000,0.0000,0.0,0.6041,0.0,0.6041,0.0000,1.0000,0.8278
fastest,0.4111,0.6389,-0.2278,0.6435,0.2667,0.2667,1.0,0.3295,0.0,0.3295,0.4545,0.0000,0.0333
fallback_relative_oracle,0.7000,0.6389,0.0611,1.0957,0.0000,0.0000,0.0,0.3331,0.0,0.3331,0.4486,0.1722,0.0000
router,0.6389,0.6389,0.0000,1.0000,0.0000,0.0000,0.0,0.6041,0.0,0.6041,0.0000,1.0000,0.8278


{'router_active': False, 'selected_thresholds': {'qwen2.5-1.5b-ar': np.float64(1.0), 'fast-dllm-v2-1.5b': np.float64(1.0)}, 'selected_latency_blend': 0.0, 'platt_parameters': {'qwen2.5-1.5b-ar': {'slope': 0.5058, 'intercept': 0.9067}, 'fast-dllm-v2-1.5b': {'slope': 1.9788, 'intercept': 0.235}}, 'mean_diagnostic_router_overhead_ms': np.float64(50.966)}


## 8. Diagnostics and paired uncertainty

The most informative comparisons are paired because every strategy is
evaluated on exactly the same prompts. The bootstrap resamples prompt
indices and reports uncertainty for accuracy difference and net latency
reduction relative to fallback.

Calibration and prediction diagnostics do not change the already-frozen
policy. They explain whether a negative result came from unreliable safety,
weak latency prediction, excessive overhead, or a lack of usable candidates.


In [10]:
test_targets = replacement_safe[test_indices]
calibration_rows = []
for column, candidate in enumerate(NONFALLBACK_NAMES):
    calibration_rows.append({
        "candidate": candidate,
        "platt_slope": final_platt_parameters[column][0],
        "platt_intercept": final_platt_parameters[column][1],
        "test_brier_score": brier_score_loss(
            test_targets[:, column], test_probability[:, column]
        ),
        "mean_predicted_safety": test_probability[:, column].mean(),
        "observed_safety": test_targets[:, column].mean(),
    })
calibration_diagnostics = pd.DataFrame(calibration_rows).set_index("candidate")
calibration_diagnostics.to_csv(
    V4_REPORTS / "calibration_diagnostics_v4.csv"
)

latency_rows = []
for model_index, model_name in enumerate(MODEL_NAMES):
    actual_latency = L[test_indices, model_index]
    predicted_latency = test_latency_prediction[:, model_index]
    latency_rows.append({
        "model": model_name,
        "test_mae_s": mean_absolute_error(
            actual_latency, predicted_latency
        ),
        "test_r2": r2_score(actual_latency, predicted_latency),
        "token_mae": mean_absolute_error(
            T[test_indices, model_index],
            test_prediction["tokens"][:, model_index],
        ),
    })
latency_diagnostics = pd.DataFrame(latency_rows).set_index("model")
latency_diagnostics.to_csv(
    V4_REPORTS / "latency_diagnostics_v4.csv"
)

row = np.arange(len(test_indices))
router_quality = Q[test_indices][row, router_test_idx]
fallback_test_quality = Q[test_indices, strongest_idx]
router_latency = (
    L[test_indices][row, router_test_idx] + deployed_test_overhead
)
fallback_test_latency = L[test_indices, strongest_idx]
rng = np.random.default_rng(SEED)
bootstrap_rows = []
for _ in range(2000):
    sample = rng.integers(0, len(test_indices), len(test_indices))
    bootstrap_rows.append({
        "accuracy_delta": (
            router_quality[sample].mean()
            - fallback_test_quality[sample].mean()
        ),
        "latency_reduction": 1.0 - (
            router_latency[sample].mean()
            / fallback_test_latency[sample].mean()
        ),
    })
bootstrap = pd.DataFrame(bootstrap_rows)
confidence_intervals = bootstrap.quantile([0.025, 0.5, 0.975])
confidence_intervals.to_csv(
    V4_REPORTS / "paired_bootstrap_intervals_v4.csv"
)

selected_models = np.array(MODEL_NAMES, dtype=object)[router_test_idx]
selection_by_task = pd.crosstab(
    table.loc[test_indices, "task"], selected_models,
    normalize="index",
)
eligibility_frame = pd.DataFrame({
    "task": table.loc[test_indices, "task"].to_numpy(),
    **{
        candidate: diagnostic_eligible[:, model_index]
        for candidate, model_index in zip(
            NONFALLBACK_NAMES, nonfallback_indices
        )
    },
})
eligibility_by_task = eligibility_frame.groupby("task").mean()

display(calibration_diagnostics.round(4))
display(latency_diagnostics.round(4))
display(confidence_intervals.round(4))
display(selection_by_task.round(3))
display(eligibility_by_task.round(3))


,platt_slope,platt_intercept,test_brier_score,mean_predicted_safety,observed_safety
candidate,,,,,
qwen2.5-1.5b-ar,0.5058,0.9067,0.1989,0.7732,0.7333
fast-dllm-v2-1.5b,1.9788,0.2350,0.2026,0.7348,0.7500


,test_mae_s,test_r2,token_mae
model,,,
qwen2.5-1.5b-ar,0.0781,-0.1356,1.8061
fast-dllm-v2-1.5b,0.3559,-0.0053,23.0295
qwen2.5-7b-4bit,0.0656,0.0306,1.1841


,accuracy_delta,latency_reduction
0.025,0.0,0.0
0.500,0.0,0.0
0.975,0.0,0.0


col_0,qwen2.5-7b-4bit
task,
arc_challenge,1.0
gsm8k,1.0
mmlu,1.0


,qwen2.5-1.5b-ar,fast-dllm-v2-1.5b
task,,
arc_challenge,0.0,0.0
gsm8k,0.0,0.0
mmlu,0.0,0.0


## 9. Export decisions and a reconstructable router

The decision report stores both deployed and diagnostic behavior. When the
router is disabled, `selected_model` is always the fallback, while
`diagnostic_selected_model` shows what the learned rule would have selected.
This keeps operational truth separate from research diagnosis.

The artifact contains the compact LoRA adapter, three heads, tokenizer,
calibration coefficients, thresholds, latency blend, and training-only
baseline latencies.


In [11]:
report = table.loc[test_indices, [
    "prompt_id", "task", "subject", "prompt", "reference"
]].copy().reset_index(drop=True)
report["router_active"] = router_active
report["selected_model"] = np.array(MODEL_NAMES, dtype=object)[router_test_idx]
report["diagnostic_selected_model"] = np.array(
    MODEL_NAMES, dtype=object
)[diagnostic_test_idx]
report["selected_fallback"] = router_test_idx == strongest_idx
report["diagnostic_selected_fallback"] = diagnostic_selected_fallback
report["no_eligible_alternative"] = diagnostic_no_eligible
report["quality"] = Q[test_indices][row, router_test_idx]
report["fallback_quality"] = Q[test_indices, strongest_idx]
report["quality_regret"] = np.maximum(
    0.0, report.fallback_quality - report.quality
)
report["generation_s"] = L[test_indices][row, router_test_idx]
report["router_overhead_s"] = deployed_test_overhead
report["latency_s"] = report.generation_s + report.router_overhead_s
for column, candidate in enumerate(NONFALLBACK_NAMES):
    model_index = nonfallback_indices[column]
    report[f"p_safe__{candidate}"] = test_probability[:, column]
    report[f"eligible__{candidate}"] = diagnostic_eligible[:, model_index]
for model_index, model_name in enumerate(MODEL_NAMES):
    report[f"predicted_latency__{model_name}"] = (
        test_latency_prediction[:, model_index]
    )
    report[f"measured_latency__{model_name}"] = L[
        test_indices, model_index
    ]
    report[f"quality__{model_name}"] = Q[test_indices, model_index]
report.to_parquet(
    V4_REPORTS / "test_decisions_v4.parquet", index=False
)

artifact_dir = V4_ARTIFACTS / (
    f"{EVIDENCE_FINGERPRINT[:16]}__{ROUTER_FINGERPRINT[:12]}"
)
artifact_dir.mkdir(parents=True, exist_ok=True)
router_model.encoder.save_pretrained(artifact_dir / "lora_adapter")
head_state = {
    name: value.detach().cpu()
    for name, value in router_model.state_dict().items()
    if not name.startswith("encoder.")
}
torch.save(head_state, artifact_dir / "router_heads.pt")
router_tokenizer.save_pretrained(artifact_dir / "tokenizer")

artifact_manifest = {
    **V4_ROUTER_CONTRACT,
    "artifact_schema_version": 1,
    "router_fingerprint": ROUTER_FINGERPRINT,
    "model_names": MODEL_NAMES,
    "fallback_model": fallback_name,
    "nonfallback_models": NONFALLBACK_NAMES,
    "router_active": router_active,
    "best_epoch": best_epoch,
    "selected_thresholds": dict(zip(
        NONFALLBACK_NAMES, selected_thresholds.tolist()
    )),
    "selected_latency_blend": selected_latency_blend,
    "platt_parameters": {
        candidate: {"slope": parameters[0], "intercept": parameters[1]}
        for candidate, parameters in zip(
            NONFALLBACK_NAMES, final_platt_parameters
        )
    },
    "task_model_median_latency": {
        task: {
            model: float(task_model_median.loc[task, model])
            for model in MODEL_NAMES
        }
        for task in TASKS
    },
    "router_input_template": (
        "[TASK={task}] [SUBJECT={subject}] [CHOICES={num_choices}] "
        "[LENGTH_BIN={length_bin}] {prompt}"
    ),
    "router_load_time_s": router_load_time_s,
    "router_training_time_s": router_training_time_s,
}
(artifact_dir / "router_manifest.json").write_text(
    json.dumps(artifact_manifest, indent=2), encoding="utf-8"
)
print("Saved v4 reports to", V4_REPORTS)
print("Saved v4 router artifacts to", artifact_dir)


Saved v4 reports to /content/drive/MyDrive/llm_router_v3/reports_v4
Saved v4 router artifacts to /content/drive/MyDrive/llm_router_v3/artifacts_v4/d2044a470f45f714__25c4fd832dd9


## 10. Interactive decision inspector

Use the inspector to understand why an alternative was accepted or rejected.
It shows calibrated safety, threshold eligibility, predicted and measured
latency, observed quality, and the distinction between diagnostic and
deployed selection.


In [12]:
import ipywidgets as widgets

task_rows = {
    task: frame.sort_values("prompt_id").reset_index(drop=True)
    for task, frame in report.groupby("task", sort=True)
}
task_selector = widgets.Dropdown(
    options=list(task_rows), description="Task:",
    layout=widgets.Layout(width="340px"),
)
prompt_selector = widgets.IntSlider(
    value=1, min=1, max=len(task_rows[task_selector.value]), step=1,
    description="Prompt:", continuous_update=False,
    layout=widgets.Layout(width="520px"),
)
previous_button = widgets.Button(description="Previous")
next_button = widgets.Button(description="Next", button_style="primary")
inspector_output = widgets.Output()

def render_decision(*_):
    frame = task_rows[task_selector.value]
    position = min(prompt_selector.value - 1, len(frame) - 1)
    decision = frame.iloc[position]
    rows = []
    for model_index, model_name in enumerate(MODEL_NAMES):
        if model_index == strongest_idx:
            probability = "always available"
            eligible = True
        else:
            column = int(np.where(nonfallback_indices == model_index)[0][0])
            probability = f"{decision[f'p_safe__{model_name}']:.3f}"
            eligible = bool(decision[f"eligible__{model_name}"])
        rows.append(
            "<tr>"
            f"<td><strong>{html_lib.escape(model_name)}</strong></td>"
            f"<td>{probability}</td><td>{eligible}</td>"
            f"<td>{decision[f'predicted_latency__{model_name}']:.3f} s</td>"
            f"<td>{decision[f'measured_latency__{model_name}']:.3f} s</td>"
            f"<td>{int(decision[f'quality__{model_name}'])}</td>"
            "</tr>"
        )

    card = f"""
    <div style="font-family:Inter,system-ui,sans-serif;border:1px solid #dbe3ef;
                border-radius:16px;padding:20px;background:#f8fbff;color:#172033">
      <div><strong>{html_lib.escape(str(decision.task))}</strong> ·
        Router active: {bool(decision.router_active)} ·
        No eligible alternative: {bool(decision.no_eligible_alternative)}</div>
      <h3>Deployed: {html_lib.escape(str(decision.selected_model))}</h3>
      <p>Diagnostic choice: {html_lib.escape(str(decision.diagnostic_selected_model))}</p>
      <pre style="white-space:pre-wrap;background:white;padding:12px;border-radius:8px">{html_lib.escape(str(decision.prompt))}</pre>
      <p><strong>Reference:</strong> {html_lib.escape(str(decision.reference))}</p>
      <table style="border-collapse:collapse;width:100%;background:white">
        <thead><tr><th>Model</th><th>P(safe)</th><th>Eligible</th>
          <th>Predicted</th><th>Measured</th><th>Quality</th></tr></thead>
        <tbody>{''.join(rows)}</tbody>
      </table>
    </div>
    """
    with inspector_output:
        clear_output(wait=True)
        display(HTML(card))

def reset_task(change):
    prompt_selector.max = len(task_rows[change["new"]])
    prompt_selector.value = 1
    render_decision()

task_selector.observe(reset_task, names="value")
prompt_selector.observe(render_decision, names="value")
previous_button.on_click(
    lambda _: setattr(
        prompt_selector, "value", max(1, prompt_selector.value - 1)
    )
)
next_button.on_click(
    lambda _: setattr(
        prompt_selector, "value",
        min(prompt_selector.max, prompt_selector.value + 1),
    )
)
display(widgets.VBox([
    widgets.HBox([task_selector, previous_button, next_button]),
    prompt_selector,
    inspector_output,
]))
render_decision()


## 11. How to interpret v4

A positive result requires all synchronized conditions:

1. sealed-test quality retention is at least 0.98;
2. net latency reduction is positive after router overhead;
3. the paired latency interval does not materially support a slowdown;
4. non-fallback selections occur in defensible subgroups; and
5. a faster worker is used often enough to justify maintaining it.

If the router is disabled again, use the diagnostic tables to locate the
bottleneck:

- poor Brier scores indicate unreliable safety prediction;
- weak latency MAE/R² indicates that task/model medians or better targets are
  preferable;
- many eligible alternatives but no net gain points to router overhead;
- few safe-faster training examples points to the candidate pool itself;
- a strong fallback-relative oracle with a weak router points to insufficient
  data or features rather than absent routing opportunity.

These results remain controlled warm batch-size-one measurements on one GPU.
Production savings require already-loaded candidate workers.
